# Day G7 — 평가·편향·설명·규제, 그리고 정직한 미니 프로젝트

설명은 학습 사이트(또는 원본 레슨)에서 읽고, 코드는 이 노트북에서 위에서 아래로 차례대로 실행하세요. 앞 셀이 만든 변수와 파일을 뒤 셀이 씁니다.

원본 레슨: [lessons/advanced/G-ai-biomed/day-G7.md](https://github.com/evansuhohyun-927/neurodegeneration-bioinformatics/blob/main/lessons/advanced/G-ai-biomed/day-G7.md)

In [ ]:
# 레슨 코드가 쓰는 폴더를 미리 만든다 (Colab에서는 /content 아래에 생긴다)
import os
for d in ["data", "figures", "reports", "results", "../data", "../figures", "../reports"]:
    os.makedirs(d, exist_ok=True)

## 예시로 이해하기 — 미니 프로젝트

> 두 사이트의 **가상** 참가자 데이터입니다. 사이트 B는 표본이 작고, 기기 차이로 cadence 값이 체계적으로 밀려 있게 만들었습니다. 1명 = 1행이라 행 단위 분할이 곧 참가자 단위 분할입니다(반복 측정이면 GroupKFold, Day 33).

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, brier_score_loss, recall_score
from sklearn.inspection import permutation_importance
from sklearn.mixture import GaussianMixture

rng = np.random.default_rng(42)
FEATS = ["stride_cv", "cadence", "arm_swing", "age"]

def site(n, name, shift, prevalence):
    """가상 참가자 표(1행 = 1명). 사이트 B는 기기 차이로 cadence 값이 체계적으로 밀려 있다."""
    y = (rng.random(n) < prevalence).astype(int)
    df = pd.DataFrame({
        "stride_cv": rng.normal(2.0 + 1.0 * y, 0.7, n),
        "cadence": rng.normal(112 - 6 * y + shift, 7, n),
        "arm_swing": rng.normal(1.0 - 0.25 * y, 0.25, n),
        "age": rng.normal(66, 8, n),
        "site": name, "y": y})
    return df

df = pd.concat([site(400, "A", 0, 0.4), site(80, "B", 8, 0.25)], ignore_index=True)
df["pid"] = np.arange(len(df))                      # 참가자 ID (1명 1행이라 행 = 참가자)
tr, te = train_test_split(df, test_size=0.4, random_state=0, stratify=df[["site", "y"]])

def fit(train):
    return make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(train[FEATS], train["y"])

def boot_auc(y, p, n=1000):
    idx = [rng.integers(0, len(y), len(y)) for _ in range(n)]
    vals = [roc_auc_score(y[i], p[i]) for i in idx if len(set(y[i])) == 2]
    return np.percentile(vals, [2.5, 97.5])

def evaluate(model, label):
    p = model.predict_proba(te[FEATS])[:, 1]
    thr = 0.5
    rows = []
    for name, sel in [("전체", np.ones(len(te), bool)), ("사이트 A", te.site.values == "A"), ("사이트 B", te.site.values == "B")]:
        yy, pp = te.y.values[sel], p[sel]
        lo, hi = boot_auc(yy, pp)
        rows.append({"모델": label, "집단": name, "n": int(sel.sum()), "양성": int(yy.sum()),
                     "AUC": round(roc_auc_score(yy, pp), 3), "AUC 95% CI": f"{lo:.2f}–{hi:.2f}",
                     "민감도@0.5": round(recall_score(yy, pp >= thr), 2),
                     "Brier": round(brier_score_loss(yy, pp), 3),
                     "평균 예측 vs 실제 비율": f"{pp.mean():.2f} vs {yy.mean():.2f}"})
    return pd.DataFrame(rows), p

base = fit(tr)
t1, p_base = evaluate(base, "기준")

# 합성 증강은 '학습 데이터 안에서만': 사이트 B 학습 행으로만 GMM을 맞춰 B를 늘린다
trB = tr[tr.site == "B"]
aug = [tr]
for y, g in trB.groupby("y"):
    gm = GaussianMixture(n_components=1, random_state=0).fit(g[FEATS])
    s, _ = gm.sample(len(g) * 3)
    aug.append(pd.DataFrame(s, columns=FEATS).assign(site="B", y=y))
t2, _ = evaluate(fit(pd.concat(aug, ignore_index=True)), "B 합성 증강")
table = pd.concat([t1, t2], ignore_index=True)
pd.set_option("display.width", 200)
print(table.to_string(index=False))

imp = permutation_importance(base, te[FEATS], te["y"], scoring="roc_auc", n_repeats=30, random_state=0)
print("\n순열 중요도(AUC 감소량, 평균 ± 표준편차):")
for f, m, s in sorted(zip(FEATS, imp.importances_mean, imp.importances_std), key=lambda x: -x[1]):
    print(f"  {f:<10} {m:.3f} ± {s:.3f}")

# 모델 카드 초안을 파일로 저장한다
out = Path("reports/advanced"); out.mkdir(parents=True, exist_ok=True)
card = f"""# 모델 카드 — G7 가상 보행 분류 모델 (연습용)

- **용도:** 교육용 연습. 실제 진단·선별에 쓰지 않는다.
- **데이터:** 가상 참가자 {len(df)}명(사이트 A {int((df.site=='A').sum())}명, B {int((df.site=='B').sum())}명), 1명 1행, 참가자 단위 6:4 분할.
- **모델:** 표준화 + 로지스틱 회귀, feature {', '.join(FEATS)}.
- **평가(평가셋):**

{table.to_markdown(index=False) if hasattr(table, 'to_markdown') else table.to_string(index=False)}

- **알려진 한계:** 사이트 B는 표본이 작아 신뢰구간이 넓다. 사이트 B의 cadence는 기기 차이로 밀려 있다(가상 설정). 합성 증강은 학습셋 안에서만 했다.
- **하지 말 것:** 다른 기기·다른 인구집단에 그대로 적용, 성능 숫자를 진단 정확도로 표현.
"""
(out / "G7_model_card.md").write_text(card, encoding="utf-8")
print("\n저장:", out / "G7_model_card.md")